# 🧠 Intelligent Data Analyzer
> **Plug-and-play automated data science toolkit** — Upload any CSV and get instant AI-assisted insights, anomaly detection, automated ML, and a downloadable report.

**Author:** Your Name | **GitHub:** your-github | **Live Demo:** your-streamlit-link

---
### 📦 Stack
- **Data:** `pandas`, `numpy`
- **Visualization:** `matplotlib`, `seaborn`, `plotly`
- **ML / AutoML:** `scikit-learn`, `xgboost`, `lightgbm`, `pycaret`
- **Stats:** `scipy`, `statsmodels`
- **Reports:** `fpdf2`, `jinja2`
- **Anomaly Detection:** `IsolationForest`, `LOF`
- **AI Features:** OpenAI GPT or local heuristics for feature suggestions
---

## ⚙️ STEP 0 — Install All Dependencies

In [1]:
# Run once — installs everything needed
import subprocess, sys

packages = [
    'pandas', 'numpy', 'matplotlib', 'seaborn', 'plotly',
    'scikit-learn', 'xgboost', 'lightgbm', 'scipy', 'statsmodels',
    'fpdf2', 'jinja2', 'ipywidgets', 'kaleido', 'pycaret',
    'colorama', 'tabulate'
]

for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

print('✅ All packages installed successfully!')

✅ All packages installed successfully!


## 📥 STEP 1 — Load Dataset

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────────────────────
# 🔧 CONFIG — Edit this section before running
# ─────────────────────────────────────────────────────────────
CSV_PATH    = 'your_dataset.csv'   # ← Path to your CSV file
TARGET_COL  = 'target'             # ← Name of target/label column (or None for EDA only)
TASK_TYPE   = 'auto'               # ← 'classification', 'regression', or 'auto'
REPORT_NAME = 'eda_report'         # ← Output report filename (no extension)
# ─────────────────────────────────────────────────────────────

# ── Load ──
df = pd.read_csv(CSV_PATH)

print(f'✅ Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'📋 Columns: {list(df.columns)}')
df.head()

## 🔍 STEP 2 — Automated Data Profiling

In [ ]:
from tabulate import tabulate
from IPython.display import display, HTML

def profile_dataset(df):
    """Generate a comprehensive automated data profile."""
    
    print('\n' + '═'*60)
    print('  📊  DATASET PROFILE')
    print('═'*60)
    
    # ── Basic Info ──
    print(f'\n📐 Shape          : {df.shape[0]:,} rows × {df.shape[1]} columns')
    print(f'💾 Memory Usage   : {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB')
    print(f'🔢 Numeric Cols   : {len(df.select_dtypes(include=np.number).columns)}')
    print(f'🔤 Categorical Cols: {len(df.select_dtypes(include="object").columns)}')
    print(f'📅 Datetime Cols  : {len(df.select_dtypes(include="datetime").columns)}')
    print(f'❓ Total Nulls    : {df.isnull().sum().sum():,} ({df.isnull().mean().mean()*100:.1f}% of cells)')
    print(f'🔁 Duplicate Rows : {df.duplicated().sum():,}')
    
    # ── Per-Column Summary ──
    print('\n' + '─'*60)
    print('  📋  COLUMN-LEVEL SUMMARY')
    print('─'*60)
    
    rows = []
    for col in df.columns:
        dtype     = str(df[col].dtype)
        nulls     = df[col].isnull().sum()
        null_pct  = f'{nulls/len(df)*100:.1f}%'
        unique    = df[col].nunique()
        sample    = str(df[col].dropna().iloc[0]) if not df[col].dropna().empty else 'N/A'
        rows.append([col, dtype, nulls, null_pct, unique, sample[:30]])
    
    print(tabulate(rows,
                   headers=['Column','DType','Nulls','Null%','Unique','Sample'],
                   tablefmt='rounded_outline'))
    
    # ── Numeric Stats ──
    num_df = df.select_dtypes(include=np.number)
    if not num_df.empty:
        print('\n' + '─'*60)
        print('  📈  NUMERIC STATISTICS')
        print('─'*60)
        stats = num_df.describe().T
        stats['skewness'] = num_df.skew()
        stats['kurtosis'] = num_df.kurtosis()
        display(stats.style.background_gradient(cmap='Blues').format('{:.2f}'))
    
    return df.isnull().sum()

null_summary = profile_dataset(df)

## 🧹 STEP 3 — Automated Data Cleaning & Preprocessing

In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer

def auto_clean(df, target_col=None):
    """Automated cleaning: handle nulls, duplicates, types, encoding."""
    df_clean = df.copy()
    report   = []
    
    # 1️⃣ Remove duplicates
    before = len(df_clean)
    df_clean.drop_duplicates(inplace=True)
    removed = before - len(df_clean)
    if removed:
        report.append(f'🗑️  Removed {removed:,} duplicate rows')
    
    # 2️⃣ Parse dates (auto-detect columns with 'date' or 'time' in name)
    for col in df_clean.columns:
        if any(kw in col.lower() for kw in ['date','time','dt','created','updated']):
            try:
                df_clean[col] = pd.to_datetime(df_clean[col])
                df_clean[f'{col}_year']  = df_clean[col].dt.year
                df_clean[f'{col}_month'] = df_clean[col].dt.month
                df_clean[f'{col}_dow']   = df_clean[col].dt.dayofweek
                df_clean.drop(columns=[col], inplace=True)
                report.append(f'📅 Extracted year/month/dow from `{col}`')
            except:
                pass
    
    # 3️⃣ Impute numeric: median
    num_cols = df_clean.select_dtypes(include=np.number).columns.tolist()
    if target_col and target_col in num_cols:
        num_cols.remove(target_col)
    if num_cols:
        imp = SimpleImputer(strategy='median')
        df_clean[num_cols] = imp.fit_transform(df_clean[num_cols])
        report.append(f'🔢 Imputed {len(num_cols)} numeric cols with median')
    
    # 4️⃣ Impute categorical: mode
    cat_cols = df_clean.select_dtypes(include='object').columns.tolist()
    if target_col and target_col in cat_cols:
        cat_cols.remove(target_col)
    for col in cat_cols:
        df_clean[col].fillna(df_clean[col].mode()[0], inplace=True)
    if cat_cols:
        report.append(f'🔤 Imputed {len(cat_cols)} categorical cols with mode')
    
    # 5️⃣ Drop high-cardinality / high-null columns
    to_drop = []
    for col in df_clean.columns:
        if col == target_col: continue
        null_pct    = df_clean[col].isnull().mean()
        cardinality = df_clean[col].nunique() / len(df_clean)
        if null_pct > 0.6:
            to_drop.append(col)
            report.append(f'⚠️  Dropped `{col}` (>{int(null_pct*100)}% null)')
        elif df_clean[col].dtype == 'object' and cardinality > 0.95:
            to_drop.append(col)
            report.append(f'⚠️  Dropped `{col}` (high-cardinality ID-like column)')
    df_clean.drop(columns=to_drop, inplace=True)
    
    # 6️⃣ Label-encode remaining categoricals
    cat_cols = df_clean.select_dtypes(include='object').columns.tolist()
    if target_col and target_col in cat_cols:
        cat_cols.remove(target_col)
    le = LabelEncoder()
    for col in cat_cols:
        df_clean[col] = le.fit_transform(df_clean[col].astype(str))
    if cat_cols:
        report.append(f'🏷️  Label-encoded {len(cat_cols)} categorical columns')
    
    # ── Print Report ──
    print('\n' + '═'*60)
    print('  🧹  AUTO-CLEANING REPORT')
    print('═'*60)
    for step in report:
        print(f'  {step}')
    print(f'\n✅ Clean shape: {df_clean.shape[0]:,} rows × {df_clean.shape[1]} columns')
    
    return df_clean


df_clean = auto_clean(df, TARGET_COL)
df_clean.head()

## 📊 STEP 4 — Automated EDA Visualizations

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── Matplotlib style ──
plt.rcParams.update({
    'figure.facecolor' : '#0f0f0f',
    'axes.facecolor'   : '#1a1a2e',
    'axes.edgecolor'   : '#444',
    'text.color'       : 'white',
    'axes.labelcolor'  : 'white',
    'xtick.color'      : '#aaa',
    'ytick.color'      : '#aaa',
    'grid.color'       : '#333',
    'grid.alpha'       : 0.3,
})

PALETTE = ['#7c3aed','#2563eb','#0891b2','#059669','#d97706','#dc2626','#ec4899']

num_cols = df_clean.select_dtypes(include=np.number).columns.tolist()
cat_cols = df_clean.select_dtypes(exclude=np.number).columns.tolist()

# Remove target if present
feat_num = [c for c in num_cols if c != TARGET_COL]


# ── 4A: Distribution Grid ──
def plot_distributions(df, cols, max_cols=4):
    cols_to_plot = cols[:12]  # limit to 12
    n = len(cols_to_plot)
    rows = (n // max_cols) + (1 if n % max_cols else 0)
    fig, axes = plt.subplots(rows, max_cols, figsize=(18, rows * 3.5))
    axes = axes.flatten() if n > 1 else [axes]
    
    for i, col in enumerate(cols_to_plot):
        ax = axes[i]
        ax.hist(df[col].dropna(), bins=30, color=PALETTE[i % len(PALETTE)],
                alpha=0.85, edgecolor='none')
        ax.set_title(col, fontsize=10, color='white', pad=6)
        ax.set_xlabel('')
        skew_val = df[col].skew()
        ax.text(0.98, 0.95, f'skew={skew_val:.2f}', transform=ax.transAxes,
                ha='right', va='top', fontsize=8, color='#aaa')
        ax.grid(True, alpha=0.2)
    
    for j in range(i+1, len(axes)):
        axes[j].set_visible(False)
    
    fig.suptitle('📊 Feature Distributions', fontsize=16, color='white', y=1.01)
    plt.tight_layout()
    plt.savefig('distributions.png', dpi=150, bbox_inches='tight',
                facecolor='#0f0f0f')
    plt.show()
    print('💾 Saved: distributions.png')

if feat_num:
    plot_distributions(df_clean, feat_num)

In [ ]:
# ── 4B: Correlation Heatmap (Interactive Plotly) ──

def plot_correlation_heatmap(df, max_cols=20):
    num = df.select_dtypes(include=np.number)
    if num.shape[1] > max_cols:
        # keep top correlated with target if exists
        if TARGET_COL in num.columns:
            top = num.corr()[TARGET_COL].abs().nlargest(max_cols).index
            num = num[top]
        else:
            num = num.iloc[:, :max_cols]
    
    corr = num.corr()
    
    fig = go.Figure(go.Heatmap(
        z=corr.values,
        x=corr.columns.tolist(),
        y=corr.index.tolist(),
        colorscale='RdBu_r',
        zmid=0,
        text=np.round(corr.values, 2),
        texttemplate='%{text}',
        textfont={'size': 9},
        hovertemplate='%{x} × %{y}: %{z:.3f}<extra></extra>',
    ))
    
    fig.update_layout(
        title='🔥 Correlation Heatmap',
        template='plotly_dark',
        width=800, height=600,
        font=dict(size=10),
    )
    fig.show()
    fig.write_html('correlation_heatmap.html')
    print('💾 Saved: correlation_heatmap.html')

plot_correlation_heatmap(df_clean)

In [ ]:
# ── 4C: Target Analysis ──

def analyze_target(df, target_col):
    if target_col not in df.columns:
        print(f'⚠️  Target column `{target_col}` not found. Skipping.')
        return
    
    target = df[target_col]
    is_classification = target.nunique() <= 20 or target.dtype == 'object'
    
    if is_classification:
        counts = target.value_counts()
        fig = px.bar(x=counts.index.astype(str), y=counts.values,
                     labels={'x': target_col, 'y': 'Count'},
                     title=f'🎯 Target Distribution: {target_col}',
                     color=counts.values,
                     color_continuous_scale='Viridis',
                     template='plotly_dark')
        fig.update_layout(showlegend=False)
        fig.show()
        
        # Class balance warning
        imbalance = counts.max() / counts.min()
        if imbalance > 3:
            print(f'⚠️  Class imbalance detected! Ratio: {imbalance:.1f}x')
            print('   💡 Consider: SMOTE, class_weight="balanced", or oversampling.')
    else:
        fig = px.histogram(df, x=target_col, nbins=50,
                           title=f'🎯 Target Distribution: {target_col}',
                           template='plotly_dark',
                           color_discrete_sequence=['#7c3aed'])
        fig.show()
        print(f'  Mean:   {target.mean():.4f}')
        print(f'  Median: {target.median():.4f}')
        print(f'  Std:    {target.std():.4f}')
        print(f'  Skew:   {target.skew():.4f}')

if TARGET_COL:
    analyze_target(df_clean, TARGET_COL)

In [ ]:
# ── 4D: Interactive Pairplot (Plotly Scatter Matrix) ──

def interactive_pairplot(df, target_col, max_features=6):
    num = df.select_dtypes(include=np.number)
    features = [c for c in num.columns if c != target_col][:max_features]
    if target_col and target_col in df.columns:
        features += [target_col]
    
    color_col = target_col if (target_col and target_col in df.columns and
                               df[target_col].nunique() <= 15) else None
    
    fig = px.scatter_matrix(
        df[features].dropna().sample(min(2000, len(df))),
        dimensions=[c for c in features if c != target_col],
        color=color_col,
        title='🔭 Interactive Feature Scatter Matrix',
        template='plotly_dark',
        opacity=0.5,
        color_continuous_scale='Viridis'
    )
    fig.update_traces(marker=dict(size=3))
    fig.update_layout(width=900, height=800)
    fig.show()
    fig.write_html('pairplot.html')
    print('💾 Saved: pairplot.html')

interactive_pairplot(df_clean, TARGET_COL)

## 🚨 STEP 5 — Anomaly Detection (Fraud / Outlier Detection)

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler

def detect_anomalies(df, target_col=None, contamination=0.05):
    """Dual-method anomaly detection using IsolationForest + LOF."""
    
    print('\n' + '═'*60)
    print('  🚨  ANOMALY DETECTION')
    print('═'*60)
    
    num = df.select_dtypes(include=np.number)
    if target_col and target_col in num.columns:
        num = num.drop(columns=[target_col])
    
    scaler = StandardScaler()
    X = scaler.fit_transform(num.fillna(num.median()))
    
    # ── Isolation Forest ──
    iso = IsolationForest(contamination=contamination, random_state=42, n_jobs=-1)
    iso_labels = iso.fit_predict(X)
    iso_scores = iso.score_samples(X)
    
    # ── Local Outlier Factor ──
    lof = LocalOutlierFactor(n_neighbors=20, contamination=contamination)
    lof_labels = lof.fit_predict(X)
    lof_scores = lof.negative_outlier_factor_
    
    df_result = df.copy()
    df_result['anomaly_iso']    = (iso_labels == -1).astype(int)
    df_result['anomaly_lof']    = (lof_labels == -1).astype(int)
    df_result['anomaly_score']  = -iso_scores  # higher = more anomalous
    df_result['is_anomaly']     = ((iso_labels == -1) & (lof_labels == -1)).astype(int)
    
    n_iso   = (iso_labels == -1).sum()
    n_lof   = (lof_labels == -1).sum()
    n_both  = df_result['is_anomaly'].sum()
    
    print(f'\n  🔍 IsolationForest anomalies  : {n_iso:,} ({n_iso/len(df)*100:.1f}%)')
    print(f'  🔍 LocalOutlierFactor anomalies: {n_lof:,} ({n_lof/len(df)*100:.1f}%)')
    print(f'  ⚠️  Confirmed (both methods)   : {n_both:,} ({n_both/len(df)*100:.1f}%)')
    
    # ── Interactive Plot ──
    if num.shape[1] >= 2:
        # Use first 2 PCA components for visualization
        from sklearn.decomposition import PCA
        pca = PCA(n_components=2)
        coords = pca.fit_transform(X)
        
        plot_df = pd.DataFrame({
            'PC1'         : coords[:, 0],
            'PC2'         : coords[:, 1],
            'anomaly'     : df_result['is_anomaly'].map({0: 'Normal', 1: '⚠️ Anomaly'}),
            'score'       : df_result['anomaly_score'],
        })
        
        fig = px.scatter(
            plot_df, x='PC1', y='PC2',
            color='anomaly',
            size='score',
            size_max=12,
            color_discrete_map={'Normal': '#2563eb', '⚠️ Anomaly': '#dc2626'},
            title=f'🚨 Anomaly Detection (PCA View) — {n_both} confirmed anomalies',
            template='plotly_dark',
            hover_data={'PC1': ':.3f', 'PC2': ':.3f', 'score': ':.4f'},
            opacity=0.7,
        )
        fig.update_layout(width=850, height=550)
        fig.show()
        fig.write_html('anomaly_detection.html')
        print('💾 Saved: anomaly_detection.html')
    
    # ── Top anomalous rows ──
    print('\n  🔝 Top 10 Most Anomalous Rows:')
    display(df_result.nlargest(10, 'anomaly_score')[list(num.columns[:5]) + ['anomaly_score']])
    
    return df_result

df_anomaly = detect_anomalies(df_clean, TARGET_COL)

## 🤖 STEP 6 — AI-Assisted Feature Importance & Suggestions

In [ ]:
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.inspection import permutation_importance
import scipy.stats as stats

def ai_feature_analysis(df, target_col, task_type='auto'):
    """AI-assisted feature importance with actionable recommendations."""
    
    if not target_col or target_col not in df.columns:
        print('⚠️  No target column — running unsupervised feature variance analysis.')
        variance_scores = df.select_dtypes(include=np.number).var().sort_values(ascending=False)
        print('\n📊 Top features by variance:')
        print(variance_scores.head(10))
        return
    
    X = df.drop(columns=[target_col]).select_dtypes(include=np.number).fillna(0)
    y = df[target_col]
    
    # Auto-detect task
    if task_type == 'auto':
        task_type = 'classification' if y.nunique() <= 20 else 'regression'
    
    print(f'\n  🤖 Detected task type: {task_type.upper()}')
    
    # ── Random Forest Feature Importance ──
    if task_type == 'classification':
        model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    else:
        model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    
    model.fit(X, y)
    
    importance_df = pd.DataFrame({
        'feature'    : X.columns,
        'importance' : model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    # ── Mutual Information ──
    from sklearn.feature_selection import mutual_info_classif, mutual_info_regression
    if task_type == 'classification':
        mi_scores = mutual_info_classif(X, y, random_state=42)
    else:
        mi_scores = mutual_info_regression(X, y, random_state=42)
    
    importance_df['mutual_info'] = mi_scores
    importance_df['combined_score'] = (
        importance_df['importance'] / importance_df['importance'].max() * 0.6 +
        importance_df['mutual_info'] / (importance_df['mutual_info'].max() + 1e-9) * 0.4
    )
    importance_df = importance_df.sort_values('combined_score', ascending=False)
    
    # ── Print Top 3 AI Suggestions ──
    print('\n' + '═'*60)
    print('  🧠  AI FEATURE SUGGESTIONS')
    print('═'*60)
    print(f'  Target: `{target_col}` ({task_type})')
    print()
    
    top3 = importance_df.head(3)
    for i, (_, row) in enumerate(top3.iterrows(), 1):
        corr_val = df[row['feature']].corr(y) if y.dtype in [float, int] else 'N/A'
        corr_str = f'{corr_val:.3f}' if isinstance(corr_val, float) else corr_val
        print(f'  #{i} 🏆 `{row["feature"]}`')
        print(f'      RF Importance  : {row["importance"]*100:.2f}%')
        print(f'      Mutual Info    : {row["mutual_info"]:.4f}')
        print(f'      Correlation    : {corr_str}')
        print()
    
    # ── Low-importance features to drop ──
    weak_features = importance_df[importance_df['combined_score'] < 0.01]['feature'].tolist()
    if weak_features:
        print(f'  💡 Low-impact features (consider dropping):')
        print(f'     {weak_features[:5]}')
    
    # ── Interactive Bar Chart ──
    fig = px.bar(
        importance_df.head(15),
        x='combined_score', y='feature',
        orientation='h',
        color='combined_score',
        color_continuous_scale='Viridis',
        title='🧠 AI Feature Importance (RF + Mutual Info combined)',
        labels={'combined_score': 'Combined Score', 'feature': 'Feature'},
        template='plotly_dark'
    )
    fig.update_layout(yaxis={'categoryorder': 'total ascending'}, width=800, height=500)
    fig.show()
    fig.write_html('feature_importance.html')
    print('💾 Saved: feature_importance.html')
    
    return importance_df


importance_df = ai_feature_analysis(df_clean, TARGET_COL, TASK_TYPE)

## 📐 STEP 7 — Statistical Analysis

In [ ]:
import scipy.stats as stats
import statsmodels.api as sm

def statistical_analysis(df, target_col=None):
    """Normality tests, ANOVA, and regression summary."""
    
    print('\n' + '═'*60)
    print('  📐  STATISTICAL ANALYSIS')
    print('═'*60)
    
    num_cols = df.select_dtypes(include=np.number).columns.tolist()
    feat_cols = [c for c in num_cols if c != target_col]
    
    # ── Normality Test (Shapiro-Wilk for small, D'Agostino for large) ──
    print('\n  🧪 Normality Tests:')
    norm_rows = []
    for col in feat_cols[:8]:
        sample = df[col].dropna().sample(min(5000, len(df)), random_state=42)
        if len(sample) < 8:
            continue
        if len(sample) <= 2000:
            stat, p = stats.shapiro(sample)
            test = 'Shapiro-Wilk'
        else:
            stat, p = stats.normaltest(sample)
            test = "D'Agostino"
        normal = '✅ Normal' if p > 0.05 else '❌ Non-normal'
        norm_rows.append([col, test, f'{stat:.4f}', f'{p:.4e}', normal])
    
    print(tabulate(norm_rows,
                   headers=['Feature','Test','Stat','p-value','Result'],
                   tablefmt='rounded_outline'))
    
    # ── OLS Regression Summary ──
    if target_col and target_col in df.columns:
        y = df[target_col]
        X = df[feat_cols[:8]].fillna(0)
        
        # Only run if target is numeric
        if y.dtype in [float, int, np.float64, np.int64] and y.nunique() > 10:
            X_const = sm.add_constant(X)
            try:
                ols_model = sm.OLS(y, X_const).fit()
                print('\n  📈 OLS Regression Summary (Top 8 Features → Target):')
                print(ols_model.summary().tables[0].as_text())
                print(f'  R² = {ols_model.rsquared:.4f} | Adj. R² = {ols_model.rsquared_adj:.4f}')
                print(f'  F-statistic: {ols_model.fvalue:.2f} (p = {ols_model.f_pvalue:.4e})')
                
                # ── Significant predictors ──
                sig = ols_model.pvalues[ols_model.pvalues < 0.05].index.tolist()
                sig = [s for s in sig if s != 'const']
                if sig:
                    print(f'  💡 Statistically significant predictors (p<0.05): {sig}')
            except Exception as e:
                print(f'  ⚠️  OLS failed: {e}')

statistical_analysis(df_clean, TARGET_COL)

## 🏆 STEP 8 — AutoML Pipeline (Scikit-learn + XGBoost + LightGBM)

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier
from sklearn.svm import SVC, SVR
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, mean_squared_error, r2_score, mean_absolute_error
)
import xgboost as xgb
import lightgbm as lgb
import time

def automl_pipeline(df, target_col, task_type='auto'):
    """Run multiple ML models, rank by performance, return best model."""
    
    if not target_col or target_col not in df.columns:
        print('⚠️  No target column. Skipping AutoML.')
        return None, None
    
    X = df.drop(columns=[target_col]).select_dtypes(include=np.number).fillna(0)
    y = df[target_col]
    
    if task_type == 'auto':
        task_type = 'classification' if y.nunique() <= 20 else 'regression'
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42,
        stratify=y if task_type == 'classification' else None
    )
    
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s  = scaler.transform(X_test)
    
    # ── Model Registry ──
    if task_type == 'classification':
        models = {
            'Logistic Regression'     : LogisticRegression(max_iter=1000, random_state=42),
            'Random Forest'           : RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
            'XGBoost'                 : xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss',
                                                           random_state=42, verbosity=0),
            'LightGBM'                : lgb.LGBMClassifier(random_state=42, verbose=-1),
            'Gradient Boosting'       : GradientBoostingClassifier(random_state=42),
        }
        metric_name = 'Accuracy'
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    else:
        models = {
            'Ridge Regression'        : Ridge(),
            'Random Forest'           : RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
            'XGBoost'                 : xgb.XGBRegressor(random_state=42, verbosity=0),
            'LightGBM'                : lgb.LGBMRegressor(random_state=42, verbose=-1),
        }
        metric_name = 'R²'
        cv = KFold(n_splits=5, shuffle=True, random_state=42)
    
    print('\n' + '═'*65)
    print(f'  🏆  AUTOML PIPELINE — {task_type.upper()}')
    print('═'*65)
    print(f'  Training: {len(X_train):,} samples | Test: {len(X_test):,} samples | Features: {X.shape[1]}')
    print()
    
    results = []
    trained_models = {}
    
    for name, model in models.items():
        t0 = time.time()
        try:
            # Use scaled data for linear models, raw for tree-based
            is_linear = any(k in name.lower() for k in ['logistic','ridge','svm'])
            Xt = X_train_s if is_linear else X_train
            Xv = X_test_s  if is_linear else X_test
            
            model.fit(Xt, y_train)
            preds = model.predict(Xv)
            
            if task_type == 'classification':
                score    = accuracy_score(y_test, preds)
                cv_score = cross_val_score(model, X_train, y_train, cv=cv,
                                           scoring='accuracy', n_jobs=-1).mean()
                try:
                    if y.nunique() == 2:
                        auc = roc_auc_score(y_test, model.predict_proba(Xv)[:, 1])
                    else:
                        auc = roc_auc_score(y_test, model.predict_proba(Xv),
                                            multi_class='ovr', average='weighted')
                except:
                    auc = None
                results.append({'Model': name, 'Test Acc': score, 'CV Acc': cv_score,
                                'AUC': auc, 'Time(s)': round(time.time()-t0, 2)})
            else:
                r2   = r2_score(y_test, preds)
                rmse = np.sqrt(mean_squared_error(y_test, preds))
                mae  = mean_absolute_error(y_test, preds)
                cv_score = cross_val_score(model, X_train, y_train, cv=cv,
                                           scoring='r2', n_jobs=-1).mean()
                results.append({'Model': name, 'Test R²': r2, 'CV R²': cv_score,
                                'RMSE': rmse, 'MAE': mae, 'Time(s)': round(time.time()-t0, 2)})
            
            trained_models[name] = model
            print(f'  ✅ {name:<30} done in {time.time()-t0:.1f}s')
        except Exception as e:
            print(f'  ❌ {name:<30} failed: {e}')
    
    results_df = pd.DataFrame(results)
    
    # ── Rank & Display ──
    rank_col = 'Test Acc' if task_type == 'classification' else 'Test R²'
    results_df = results_df.sort_values(rank_col, ascending=False).reset_index(drop=True)
    results_df.index += 1
    
    print('\n  📊 Leaderboard:')
    display(results_df.style
            .background_gradient(subset=[rank_col], cmap='RdYlGn')
            .format({col: '{:.4f}' for col in results_df.select_dtypes(float).columns}))
    
    best_name  = results_df.iloc[0]['Model']
    best_model = trained_models[best_name]
    
    print(f'\n  🥇 Best Model: {best_name}')
    
    # ── Confusion Matrix / Residual Plot ──
    is_linear = any(k in best_name.lower() for k in ['logistic','ridge','svm'])
    Xv = X_test_s if is_linear else X_test
    best_preds = best_model.predict(Xv)
    
    if task_type == 'classification':
        cm = confusion_matrix(y_test, best_preds)
        fig = px.imshow(cm, text_auto=True, color_continuous_scale='Blues',
                        title=f'Confusion Matrix — {best_name}',
                        labels={'x':'Predicted','y':'Actual'},
                        template='plotly_dark')
        fig.show()
        print('\n  Classification Report:')
        print(classification_report(y_test, best_preds))
    else:
        residuals = y_test - best_preds
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=best_preds, y=residuals, mode='markers',
                                  marker=dict(color='#7c3aed', opacity=0.5, size=4),
                                  name='Residuals'))
        fig.add_hline(y=0, line_dash='dash', line_color='#dc2626')
        fig.update_layout(title=f'Residual Plot — {best_name}',
                          xaxis_title='Predicted', yaxis_title='Residual',
                          template='plotly_dark', width=800, height=400)
        fig.show()
    
    return best_model, results_df

best_model, leaderboard = automl_pipeline(df_clean, TARGET_COL, TASK_TYPE)

## 📄 STEP 9 — Auto-Generated HTML + PDF Report

In [ ]:
from jinja2 import Template
from fpdf import FPDF
import base64, io, os
from datetime import datetime

def generate_html_report(df, df_clean, leaderboard=None, target_col=None,
                          report_name='eda_report'):
    """Generate a polished HTML report with all EDA insights."""
    
    # ── Inline stats ──
    num_rows     = df.shape[0]
    num_cols_raw = df.shape[1]
    null_pct     = df.isnull().mean().mean() * 100
    dup_rows     = df.duplicated().sum()
    
    # Describe table
    desc_html = df_clean.describe().T.style.background_gradient(cmap='Blues').render()
    
    # Leaderboard
    lb_html = leaderboard.to_html(classes='table', index=True) if leaderboard is not None else '<p>No ML run.</p>'
    
    template_str = """
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <title>Intelligent Data Analyzer — Report</title>
  <link href="https://fonts.googleapis.com/css2?family=Space+Grotesk:wght@400;600;700&family=JetBrains+Mono:wght@400;600&display=swap" rel="stylesheet">
  <style>
    :root {
      --bg: #0d1117; --surface: #161b22; --border: #30363d;
      --text: #e6edf3; --muted: #8b949e; --accent: #7c3aed;
      --green: #3fb950; --red: #f85149; --blue: #58a6ff;
    }
    * { box-sizing: border-box; margin: 0; padding: 0; }
    body { background: var(--bg); color: var(--text);
           font-family: 'Space Grotesk', sans-serif; padding: 2rem; }
    h1 { font-size: 2rem; color: var(--accent); margin-bottom: 0.25rem; }
    h2 { font-size: 1.2rem; color: var(--blue); margin: 2rem 0 1rem;
         border-bottom: 1px solid var(--border); padding-bottom: 0.5rem; }
    .meta { color: var(--muted); font-size: 0.85rem; margin-bottom: 2rem; }
    .grid { display: grid; grid-template-columns: repeat(auto-fit, minmax(180px, 1fr));
            gap: 1rem; margin: 1.5rem 0; }
    .card { background: var(--surface); border: 1px solid var(--border);
            border-radius: 8px; padding: 1.25rem; text-align: center; }
    .card .value { font-size: 2rem; font-weight: 700; color: var(--accent); }
    .card .label { font-size: 0.8rem; color: var(--muted); margin-top: 0.25rem; }
    table { width: 100%; border-collapse: collapse; font-size: 0.85rem;
            font-family: 'JetBrains Mono', monospace; }
    th { background: var(--surface); padding: 0.6rem 0.8rem;
         text-align: left; color: var(--blue); border-bottom: 1px solid var(--border); }
    td { padding: 0.5rem 0.8rem; border-bottom: 1px solid var(--border); color: var(--text); }
    tr:hover td { background: var(--surface); }
    .badge { display: inline-block; padding: 0.2rem 0.6rem; border-radius: 4px;
             font-size: 0.75rem; font-weight: 600; }
    .badge-green { background: rgba(63,185,80,0.15); color: var(--green); }
    .badge-red   { background: rgba(248,81,73,0.15);  color: var(--red);   }
    footer { margin-top: 3rem; color: var(--muted); font-size: 0.8rem; text-align: center; }
    .section { background: var(--surface); border: 1px solid var(--border);
               border-radius: 10px; padding: 1.5rem; margin: 1rem 0; }
  </style>
</head>
<body>

<h1>🧠 Intelligent Data Analyzer Report</h1>
<p class="meta">Generated: {{ timestamp }} | Dataset: {{ csv_name }}</p>

<div class="grid">
  <div class="card"><div class="value">{{ num_rows }}</div><div class="label">Total Rows</div></div>
  <div class="card"><div class="value">{{ num_cols }}</div><div class="label">Columns</div></div>
  <div class="card"><div class="value">{{ null_pct }}%</div><div class="label">Null Rate</div></div>
  <div class="card"><div class="value">{{ dup_rows }}</div><div class="label">Duplicates</div></div>
  <div class="card"><div class="value">{{ target }}</div><div class="label">Target Column</div></div>
</div>

<h2>📊 Statistical Summary</h2>
<div class="section">{{ desc_html }}</div>

<h2>🏆 AutoML Leaderboard</h2>
<div class="section">{{ lb_html }}</div>

<h2>🔗 Interactive Assets</h2>
<div class="section">
  <ul style="line-height:2;">
    <li><a href="correlation_heatmap.html" style="color:var(--blue)">📈 Correlation Heatmap</a></li>
    <li><a href="feature_importance.html" style="color:var(--blue)">🧠 Feature Importance</a></li>
    <li><a href="anomaly_detection.html" style="color:var(--blue)">🚨 Anomaly Detection</a></li>
    <li><a href="pairplot.html" style="color:var(--blue)">🔭 Feature Pairplot</a></li>
  </ul>
</div>

<footer>Built with Intelligent Data Analyzer · Python · Scikit-learn · Plotly · XGBoost · LightGBM</footer>
</body>
</html>
"""
    
    tmpl = Template(template_str)
    html = tmpl.render(
        timestamp  = datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        csv_name   = CSV_PATH,
        num_rows   = f'{num_rows:,}',
        num_cols   = num_cols_raw,
        null_pct   = f'{null_pct:.1f}',
        dup_rows   = f'{dup_rows:,}',
        target     = target_col or 'None',
        desc_html  = desc_html,
        lb_html    = lb_html,
    )
    
    html_path = f'{report_name}.html'
    with open(html_path, 'w') as f:
        f.write(html)
    
    print(f'✅ HTML report saved: {html_path}')
    return html_path


def generate_pdf_report(df, df_clean, leaderboard=None,
                         report_name='eda_report'):
    """Generate a PDF summary report using FPDF2."""
    
    pdf = FPDF()
    pdf.set_auto_page_break(auto=True, margin=15)
    pdf.add_page()
    
    # ── Title ──
    pdf.set_font('Helvetica', 'B', 20)
    pdf.set_text_color(124, 58, 237)
    pdf.cell(0, 12, 'Intelligent Data Analyzer Report', ln=True, align='C')
    
    pdf.set_font('Helvetica', '', 9)
    pdf.set_text_color(100, 100, 100)
    pdf.cell(0, 6, f'Generated: {datetime.now().strftime("%Y-%m-%d %H:%M")} | File: {CSV_PATH}',
             ln=True, align='C')
    pdf.ln(5)
    
    # ── Key Stats ──
    pdf.set_font('Helvetica', 'B', 13)
    pdf.set_text_color(0, 0, 0)
    pdf.cell(0, 10, 'Dataset Overview', ln=True)
    pdf.set_font('Helvetica', '', 10)
    
    stats_lines = [
        f'Rows: {df.shape[0]:,}   |   Columns: {df.shape[1]}',
        f'Null Rate: {df.isnull().mean().mean()*100:.1f}%   |   Duplicate Rows: {df.duplicated().sum():,}',
        f'Numeric Cols: {len(df.select_dtypes(include=np.number).columns)}   |   '
        f'Categorical Cols: {len(df.select_dtypes(include="object").columns)}',
        f'Target Column: {TARGET_COL or "Not specified"}',
    ]
    for line in stats_lines:
        pdf.cell(0, 7, line, ln=True)
    pdf.ln(4)
    
    # ── AutoML Leaderboard ──
    if leaderboard is not None:
        pdf.set_font('Helvetica', 'B', 13)
        pdf.cell(0, 10, 'AutoML Leaderboard', ln=True)
        pdf.set_font('Helvetica', '', 8)
        
        col_w = 190 // len(leaderboard.columns)
        # Header
        pdf.set_fill_color(124, 58, 237)
        pdf.set_text_color(255, 255, 255)
        pdf.set_font('Helvetica', 'B', 8)
        for col_name in leaderboard.columns:
            pdf.cell(col_w, 7, str(col_name)[:14], border=1, fill=True, align='C')
        pdf.ln()
        
        # Rows
        pdf.set_text_color(0, 0, 0)
        pdf.set_font('Helvetica', '', 8)
        for i, (_, row) in enumerate(leaderboard.iterrows()):
            if i == 0:
                pdf.set_fill_color(235, 245, 255)
            else:
                pdf.set_fill_color(255, 255, 255)
            for val in row:
                txt = f'{val:.4f}' if isinstance(val, float) else str(val)[:14]
                pdf.cell(col_w, 6, txt, border=1, fill=True, align='C')
            pdf.ln()
    
    # ── Distribution image ──
    if os.path.exists('distributions.png'):
        pdf.add_page()
        pdf.set_font('Helvetica', 'B', 13)
        pdf.cell(0, 10, 'Feature Distributions', ln=True)
        pdf.image('distributions.png', w=190)
    
    pdf_path = f'{report_name}.pdf'
    pdf.output(pdf_path)
    print(f'✅ PDF report saved: {pdf_path}')
    return pdf_path


html_path = generate_html_report(df, df_clean, leaderboard, TARGET_COL, REPORT_NAME)
pdf_path  = generate_pdf_report(df, df_clean, leaderboard, REPORT_NAME)

## 🚀 STEP 10 — Streamlit App (Deployment-Ready Code)

In [ ]:
STREAMLIT_CODE = '''
# app.py — run with: streamlit run app.py
# Deploy: push to GitHub → connect to Streamlit Cloud (share.streamlit.io)

import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.ensemble import IsolationForest, RandomForestClassifier, RandomForestRegressor
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, r2_score
import xgboost as xgb
import lightgbm as lgb
import warnings
warnings.filterwarnings("ignore")

st.set_page_config(
    page_title="Intelligent Data Analyzer",
    page_icon="🧠",
    layout="wide",
    initial_sidebar_state="expanded"
)

st.markdown("""
<style>
  .main { background: #0d1117; }
  h1 { color: #7c3aed !important; }
  .stMetric label { color: #8b949e !important; }
</style>
""", unsafe_allow_html=True)

# ── Header ──
st.title("🧠 Intelligent Data Analyzer")
st.caption("Upload any CSV → instant AI-powered insights, anomaly detection & AutoML")
st.divider()

# ── Sidebar ──
with st.sidebar:
    st.header("⚙️ Configuration")
    uploaded = st.file_uploader("Upload CSV", type="csv")
    target   = st.text_input("Target Column (optional)", value="")
    task     = st.selectbox("Task Type", ["auto", "classification", "regression"])
    run_ml   = st.checkbox("Run AutoML", value=True)
    run_anom = st.checkbox("Run Anomaly Detection", value=True)
    contam   = st.slider("Anomaly Contamination", 0.01, 0.2, 0.05)

if not uploaded:
    st.info("👆 Upload a CSV file from the sidebar to get started!")
    st.stop()

df = pd.read_csv(uploaded)
target = target.strip() or None

# ── Overview ──
col1, col2, col3, col4 = st.columns(4)
col1.metric("Rows",    f"{df.shape[0]:,}")
col2.metric("Columns", df.shape[1])
col3.metric("Nulls",   f"{df.isnull().sum().sum():,}")
col4.metric("Dupes",   f"{df.duplicated().sum():,}")

tab1, tab2, tab3, tab4, tab5 = st.tabs([
    "📊 EDA", "🔥 Correlations", "🚨 Anomalies", "🤖 AutoML", "🧠 Features"
])

# ── Clean ──
@st.cache_data
def quick_clean(df, target):
    d = df.copy().drop_duplicates()
    for col in d.select_dtypes(include=np.number).columns:
        if col != target: d[col].fillna(d[col].median(), inplace=True)
    for col in d.select_dtypes(include="object").columns:
        if col != target: d[col].fillna(d[col].mode()[0], inplace=True)
        if col != target: d[col] = LabelEncoder().fit_transform(d[col].astype(str))
    return d

df_c = quick_clean(df, target)

with tab1:
    st.subheader("Statistical Summary")
    st.dataframe(df.describe(), use_container_width=True)
    num_cols = df_c.select_dtypes(include=np.number).columns.tolist()
    feat = st.selectbox("Select feature to plot", [c for c in num_cols if c != target])
    fig = px.histogram(df_c, x=feat, nbins=40, template="plotly_dark",
                       color_discrete_sequence=["#7c3aed"])
    st.plotly_chart(fig, use_container_width=True)

with tab2:
    st.subheader("Correlation Heatmap")
    num = df_c.select_dtypes(include=np.number)
    corr = num.iloc[:, :15].corr()
    fig = go.Figure(go.Heatmap(z=corr.values, x=corr.columns, y=corr.index,
                               colorscale="RdBu_r", zmid=0,
                               text=np.round(corr.values,2), texttemplate="%{text}"))
    fig.update_layout(template="plotly_dark")
    st.plotly_chart(fig, use_container_width=True)

with tab3:
    if run_anom:
        st.subheader("Anomaly Detection")
        X_a = df_c.select_dtypes(include=np.number).drop(columns=[target] if target in df_c.columns else [])
        X_s = StandardScaler().fit_transform(X_a.fillna(0))
        iso = IsolationForest(contamination=contam, random_state=42)
        labels = iso.fit_predict(X_s)
        df_c["anomaly"] = (labels == -1).astype(int)
        n_anom = df_c["anomaly"].sum()
        st.metric("Anomalies Found", f"{n_anom:,} ({n_anom/len(df_c)*100:.1f}%)")
        from sklearn.decomposition import PCA
        coords = PCA(n_components=2).fit_transform(X_s)
        plot_df = pd.DataFrame({"PC1":coords[:,0],"PC2":coords[:,1],
                                 "Status":df_c["anomaly"].map({0:"Normal",1:"Anomaly"})})
        fig = px.scatter(plot_df, x="PC1", y="PC2", color="Status",
                         color_discrete_map={"Normal":"#2563eb","Anomaly":"#dc2626"},
                         template="plotly_dark", opacity=0.6)
        st.plotly_chart(fig, use_container_width=True)
        st.dataframe(df[df_c["anomaly"]==1].head(20), use_container_width=True)

with tab4:
    if run_ml and target and target in df_c.columns:
        st.subheader("AutoML Leaderboard")
        X = df_c.drop(columns=[target]).select_dtypes(include=np.number).fillna(0)
        y = df_c[target]
        t = task if task != "auto" else ("classification" if y.nunique()<=20 else "regression")
        X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
        models = {
            "RandomForest": RandomForestClassifier(100,random_state=42) if t=="classification" else RandomForestRegressor(100,random_state=42),
            "XGBoost":      xgb.XGBClassifier(eval_metric="logloss",verbosity=0,random_state=42) if t=="classification" else xgb.XGBRegressor(verbosity=0,random_state=42),
            "LightGBM":     lgb.LGBMClassifier(verbose=-1,random_state=42) if t=="classification" else lgb.LGBMRegressor(verbose=-1,random_state=42),
        }
        rows=[]
        for name, m in models.items():
            m.fit(X_tr,y_tr); p=m.predict(X_te)
            score = accuracy_score(y_te,p) if t=="classification" else r2_score(y_te,p)
            rows.append({"Model":name,"Score":round(score,4),"Metric":"Accuracy" if t=="classification" else "R²"})
        lb = pd.DataFrame(rows).sort_values("Score",ascending=False)
        st.dataframe(lb, use_container_width=True)
        fig = px.bar(lb, x="Model", y="Score", color="Score",
                     color_continuous_scale="Viridis", template="plotly_dark")
        st.plotly_chart(fig, use_container_width=True)
    else:
        st.info("Set a target column and enable AutoML in the sidebar.")

with tab5:
    if target and target in df_c.columns:
        st.subheader("AI Feature Importance")
        X = df_c.drop(columns=[target]).select_dtypes(include=np.number).fillna(0)
        y = df_c[target]
        t = task if task != "auto" else ("classification" if y.nunique()<=20 else "regression")
        m = RandomForestClassifier(50,random_state=42) if t=="classification" else RandomForestRegressor(50,random_state=42)
        m.fit(X,y)
        fi = pd.DataFrame({"Feature":X.columns,"Importance":m.feature_importances_}).sort_values("Importance",ascending=False)
        st.markdown("### 🏆 Top 3 Features Affecting Target")
        for i,(_, row) in enumerate(fi.head(3).iterrows(),1):
            st.success(f"#{i} `{row[\"Feature\"]}` — importance: {row[\"Importance\"]*100:.2f}%")
        fig = px.bar(fi.head(15), x="Importance", y="Feature", orientation="h",
                     color="Importance", color_continuous_scale="Viridis",
                     template="plotly_dark")
        fig.update_layout(yaxis={"categoryorder":"total ascending"})
        st.plotly_chart(fig, use_container_width=True)
    else:
        st.info("Set a target column to see feature importance.")
'''

with open('app.py', 'w') as f:
    f.write(STREAMLIT_CODE)

print('✅ app.py saved! Deploy with: streamlit run app.py')
print('\n  📦 requirements.txt:')

REQUIREMENTS = """pandas>=2.0
numpy>=1.24
matplotlib>=3.7
seaborn>=0.12
plotly>=5.15
scikit-learn>=1.3
xgboost>=1.7
lightgbm>=4.0
scipy>=1.11
statsmodels>=0.14
fpdf2>=2.7
jinja2>=3.1
streamlit>=1.28
tabulate>=0.9
kaleido>=0.2
pycaret>=3.1
"""

with open('requirements.txt', 'w') as f:
    f.write(REQUIREMENTS)

print(REQUIREMENTS)

---
## ✅ All Done!

### 📂 Output Files
| File | Description |
|------|-------------|
| `eda_report.html` | Interactive HTML report |
| `eda_report.pdf` | PDF summary |
| `correlation_heatmap.html` | Interactive Plotly heatmap |
| `feature_importance.html` | AI feature importance chart |
| `anomaly_detection.html` | Anomaly scatter (PCA) |
| `pairplot.html` | Interactive scatter matrix |
| `distributions.png` | Feature distribution grid |
| `app.py` | Streamlit deployment app |
| `requirements.txt` | Python dependencies |

### 🚀 Deploy to Streamlit Cloud
```bash
# 1. Push to GitHub
git init && git add . && git commit -m 'Intelligent Data Analyzer'
git push origin main

# 2. Go to share.streamlit.io → New app → Select your repo → app.py
# 3. Done! Share the live URL in your portfolio.
```

### 📌 GitHub README badge snippet
```markdown
[![Streamlit App](https://static.streamlit.io/badges/streamlit_badge_black_white.svg)](YOUR_APP_URL)
```